# 🩺 Hypertension Prediction — End-to-End Machine Learning Workflow

**Objective:** Predict whether a person has hypertension (`Has_Hypertension`) from lifestyle, demographic and health-history features, and identify which factors matter most.

**Workflow**

| # | Step |
|---|------|
| 1 | Import libraries & load data |
| 2 | Data understanding & quality checks (missing values, duplicates) |
| 3 | Data cleaning |
| 4 | Exploratory Data Analysis (univariate, bivariate, statistical tests, correlation, outliers) |
| 5 | Feature engineering |
| 6 | Train / test split & preprocessing pipeline |
| 7 | Baseline model training & evaluation (5 algorithms) |
| 8 | Cross-validation & overfitting check |
| 9 | Feature importance & feature selection |
| 10 | Hyperparameter tuning |
| 11 | ROC / Precision-Recall curves & threshold analysis |
| 12 | Leakage check (ablation without `BP_History`) |
| 13 | Final model, saving & inference on new patients |
| 14 | Conclusions |

## 1. Importing Required Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve, average_precision_score,
                             ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

## 2. About the Dataset

The dataset contains **1,985 patient records** with lifestyle and health-history attributes and a binary outcome.

| Column | Type | Description |
|:--|:--|:--|
| **Age** | numeric | Age of the person (years) |
| **Salt_Intake** | numeric | Daily salt intake |
| **Stress_Score** | numeric (0–10) | Self-reported stress level |
| **BP_History** | categorical | Past blood-pressure status: `Normal`, `Prehypertension`, `Hypertension` |
| **Sleep_Duration** | numeric | Average sleep per night (hours) |
| **BMI** | numeric | Body-mass index |
| **Medication** | categorical | Medication type (`ACE Inhibitor`, `Beta Blocker`, `Diuretic`, `Other`); blank = no medication |
| **Family_History** | binary | Family history of hypertension (`Yes`/`No`) |
| **Exercise_Level** | ordinal | `Low`, `Moderate`, `High` |
| **Smoking_Status** | binary | `Smoker` / `Non-Smoker` |
| **Has_Hypertension** | **target** | `Yes` / `No` |

### Loading the CSV file

In [ ]:
# Works locally, in Jupyter, and in Google Colab (upload the CSV to the working directory / /content)
candidates = ["hypertension_dataset.csv",
              "/content/hypertension_dataset.csv",
              "/mnt/user-data/uploads/hypertension_dataset.csv"]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place 'hypertension_dataset.csv' next to this notebook (or in /content on Colab).")

df = pd.read_csv(DATA_PATH)
print(f"Loaded from: {DATA_PATH}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

## 3. Data Understanding & Quality Checks

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Summary of categorical columns
for col in ['BP_History', 'Medication', 'Family_History', 'Exercise_Level', 'Smoking_Status', 'Has_Hypertension']:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False), "\n")

### 3.1 Missing values & duplicates

In [ ]:
missing = pd.DataFrame({'missing_count': df.isnull().sum(),
                        'missing_%': (df.isnull().mean() * 100).round(2)})
print(missing[missing.missing_count > 0])
print(f"\nDuplicate rows: {df.duplicated().sum()}")

**Observation:** Only `Medication` has missing values (~40%). These are **not random gaps** — a blank means the person is *not taking any medication*. Dropping those rows would throw away 40% of the data, and imputing with the mode would be wrong, so we encode them as an explicit category `'None'`.

## 4. Data Cleaning

In [ ]:
df['Medication'] = df['Medication'].fillna('None')

# Encode target: Yes -> 1, No -> 0
df['target'] = df['Has_Hypertension'].map({'Yes': 1, 'No': 0})
df = df.drop(columns='Has_Hypertension')

print("Remaining missing values:", int(df.isnull().sum().sum()))
print(df['Medication'].value_counts())

In [ ]:
num_cols = ['Age', 'Salt_Intake', 'Stress_Score', 'Sleep_Duration', 'BMI']
cat_cols = ['BP_History', 'Medication', 'Family_History', 'Exercise_Level', 'Smoking_Status']

# 5. Exploratory Data Analysis (EDA)

## 5.1 Distribution of the Target Variable

In [ ]:
counts = df['target'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='target', data=df, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(['No Hypertension', 'Hypertension'])
ax.set_title('Distribution of Target Variable'); ax.set_xlabel(''); ax.set_ylabel('Count')
for p, v in zip(ax.patches, counts):
    ax.annotate(f"{v}\n({v/len(df):.1%})", (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout(); plt.show()

**Interpretation:** The target is almost perfectly balanced (~52% hypertension vs ~48% no hypertension). Unlike a typical medical dataset there is **no class imbalance**, so no resampling (SMOTE / class weights) is required and accuracy is a meaningful metric.

## 5.2 Distribution of Numerical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.ravel(), num_cols):
    sns.histplot(df[feature], kde=True, ax=ax)
    ax.axvline(df[feature].mean(), color='red', ls='--', label='mean')
    ax.axvline(df[feature].median(), color='green', ls=':', label='median')
    ax.set_title(f'Distribution of {feature} (skew = {df[feature].skew():.2f})')
    ax.legend()
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

**Interpretation:** All five numerical features are roughly symmetric (skewness ≈ 0), so no log/power transformation is needed — `StandardScaler` is sufficient. `Age` is close to uniform between 18 and 84, `Stress_Score` spans the full 0–10 range, and `Salt_Intake`, `Sleep_Duration` and `BMI` are approximately bell-shaped.

## 5.3 Distribution of Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, feature in zip(axes.ravel(), cat_cols):
    sns.countplot(x=feature, data=df, ax=ax, order=df[feature].value_counts().index)
    ax.set_title(f'Distribution of {feature}')
    ax.tick_params(axis='x', rotation=15)
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

## 5.4 Numerical Features vs. Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.ravel(), num_cols):
    sns.boxplot(x='target', y=feature, data=df, ax=ax)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['No', 'Yes'])
    ax.set_title(f'{feature} vs. Hypertension'); ax.set_xlabel('Has Hypertension')
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

print(df.groupby('target')[num_cols].agg(['mean', 'median']).round(2).T)

**Interpretation:** People with hypertension are on average **older** (~54 vs ~46 yrs), have **higher stress** (5.5 vs 4.4), **higher BMI** (26.6 vs 25.3) and **higher salt intake**, and sleep **less** (6.3 vs 6.6 h). The differences are modest and the distributions overlap heavily, so no single numeric feature separates the classes on its own.

In [ ]:
# Overlapping KDEs give a clearer picture of how each feature separates the two classes
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.ravel(), num_cols):
    sns.kdeplot(data=df, x=feature, hue='target', fill=True, common_norm=False, ax=ax,
                palette={0: '#4C9F70', 1: '#D1495B'})
    ax.set_title(f'{feature} by Hypertension status')
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

## 5.5 Categorical Features vs. Target (hypertension rate per category)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, feature in zip(axes.ravel(), cat_cols):
    rate = df.groupby(feature)['target'].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.axhline(df['target'].mean() * 100, color='red', ls='--', label='overall rate')
    ax.set_title(f'Hypertension rate by {feature}')
    ax.set_ylabel('% with hypertension'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15); ax.legend()
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

**Interpretation:**
* **`BP_History`** is by far the strongest signal: 92.8% of people with a past `Hypertension` status have hypertension, versus ~35% for `Normal` and `Prehypertension`.
* **`Smoking_Status`**: smokers 71.8% vs non-smokers 44.0%.
* **`Family_History`**: 66.2% with a family history vs 38.0% without.
* **`Medication`** and **`Exercise_Level`** hover around the overall rate (~52%), consistent with the non-significant tests above.

## 5.6 Statistical Significance Tests

In [ ]:
rows = []
# Numerical: Mann-Whitney U (non-parametric, no normality assumption)
for f in num_cols:
    g1, g0 = df.loc[df.target == 1, f], df.loc[df.target == 0, f]
    stat, p = stats.mannwhitneyu(g1, g0)
    rows.append({'Feature': f, 'Test': 'Mann-Whitney U', 'Statistic': stat, 'p-value': p})
# Categorical: Chi-square test of independence
for f in cat_cols:
    chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(df[f], df['target']))
    rows.append({'Feature': f, 'Test': 'Chi-square', 'Statistic': chi2, 'p-value': p})

tests = pd.DataFrame(rows)
tests['Significant (p<0.05)'] = tests['p-value'] < 0.05
tests['p-value'] = tests['p-value'].map('{:.2e}'.format)
tests

**Interpretation:** `Age`, `Salt_Intake`, `Stress_Score`, `Sleep_Duration`, `BMI`, `BP_History`, `Family_History` and `Smoking_Status` are all significantly associated with hypertension (p < 0.05). **`Medication` (p ≈ 0.58) and `Exercise_Level` (p ≈ 0.36) show no significant association** with the outcome in this dataset.

## 5.7 Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols + ['target']].corr(), annot=True, cmap='coolwarm', fmt='.2f', center=0, square=True)
plt.title('Correlation Matrix: Numerical Features and Target')
plt.show()

**Interpretation:** Correlations with the target are all weak-to-moderate: `Age` (+0.21), `Stress_Score` (+0.19), `BMI` (+0.15), `Salt_Intake` (+0.11) and `Sleep_Duration` (−0.12). The input features are essentially uncorrelated with each other, so there is no multicollinearity problem.

## 5.8 Outlier Detection (IQR method)

In [ ]:
out_rows = []
for f in num_cols:
    q1, q3 = df[f].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[f] < lo) | (df[f] > hi)).sum()
    out_rows.append({'Feature': f, 'Lower bound': round(lo, 2), 'Upper bound': round(hi, 2),
                     'Outliers': n_out, '% of data': round(n_out / len(df) * 100, 2)})
pd.DataFrame(out_rows)

The outliers are few and medically plausible (e.g. very high BMI, very short/long sleep), so they are **kept**. Tree-based models are robust to them, and the scaler used for linear / distance-based models is applied inside the pipeline.

# 6. Feature Engineering

Clinically-motivated features are derived from the raw columns:

* **BMI_Category** – WHO categories (Underweight / Normal / Overweight / Obese)
* **Age_Group** – life-stage bands
* **Poor_Sleep** – sleeping less than 6 hours
* **High_Stress** – stress score ≥ 7
* **Lifestyle_Risk_Score** – count of risk behaviours (smoker, low exercise, poor sleep, high stress, overweight/obese)

In [ ]:
df_fe = df.copy()
df_fe['BMI_Category'] = pd.cut(df_fe['BMI'], bins=[0, 18.5, 25, 30, 100],
                               labels=['Underweight', 'Normal', 'Overweight', 'Obese']).astype(str)
df_fe['Age_Group'] = pd.cut(df_fe['Age'], bins=[17, 34, 49, 64, 120],
                            labels=['18-34', '35-49', '50-64', '65+']).astype(str)
df_fe['Poor_Sleep'] = (df_fe['Sleep_Duration'] < 6).astype(int)
df_fe['High_Stress'] = (df_fe['Stress_Score'] >= 7).astype(int)
df_fe['Lifestyle_Risk_Score'] = ((df_fe['Smoking_Status'] == 'Smoker').astype(int)
                                 + (df_fe['Exercise_Level'] == 'Low').astype(int)
                                 + df_fe['Poor_Sleep'] + df_fe['High_Stress']
                                 + df_fe['BMI_Category'].isin(['Overweight', 'Obese']).astype(int))

df_fe[['BMI_Category', 'Age_Group', 'Poor_Sleep', 'High_Stress', 'Lifestyle_Risk_Score', 'target']].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, f in zip(axes, ['BMI_Category', 'Age_Group', 'Lifestyle_Risk_Score']):
    rate = df_fe.groupby(f)['target'].mean() * 100
    if f == 'BMI_Category': rate = rate.reindex(['Underweight', 'Normal', 'Overweight', 'Obese']).dropna()
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.set_title(f'Hypertension rate by {f}'); ax.set_ylabel('% with hypertension'); ax.set_xlabel('')
plt.tight_layout(); plt.show()

The engineered features show a clear gradient (risk rises with age group and with the lifestyle-risk score), confirming they carry signal. Their real contribution is quantified later via permutation importance.

# 7. Train/Test Split & Preprocessing Pipeline

* **Stratified 80/20 split** keeps the class ratio identical in both sets.
* A `ColumnTransformer` scales numerical features and one-hot encodes categorical ones.
* The preprocessor lives **inside** each `Pipeline`, so scaling/encoding statistics are learned only from training data (no data leakage, and it is also correct during cross-validation).

In [ ]:
num_features = num_cols + ['Lifestyle_Risk_Score']
cat_features = cat_cols + ['BMI_Category', 'Age_Group']
bin_features = ['Poor_Sleep', 'High_Stress']          # already 0/1 -> passthrough

X = df_fe[num_features + cat_features + bin_features]
y = df_fe['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("Train class ratio:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Test  class ratio:", y_test.value_counts(normalize=True).round(3).to_dict())

def make_preprocessor(num, cat, binary=()):
    return ColumnTransformer([
        ('num', StandardScaler(), list(num)),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), list(cat)),
        ('bin', 'passthrough', list(binary)),
    ])

preprocessor = make_preprocessor(num_features, cat_features, bin_features)

# 8. Baseline Model Training & Evaluation

Five algorithms of increasing complexity are compared:
Logistic Regression, K-Nearest Neighbours, Decision Tree, Random Forest and Gradient Boosting.

**Metrics:** Accuracy, Precision, Recall, F1-score and ROC-AUC. In a screening context **recall** (not missing real hypertension cases) matters a lot, so it is tracked next to precision.

In [ ]:
def build_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=15),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }

def evaluate(pipe, X_te, y_te, threshold=0.5):
    prob = pipe.predict_proba(X_te)[:, 1]
    pred = (prob >= threshold).astype(int)
    return {"Accuracy": accuracy_score(y_te, pred), "Precision": precision_score(y_te, pred),
            "Recall": recall_score(y_te, pred), "F1-Score": f1_score(y_te, pred),
            "ROC-AUC": roc_auc_score(y_te, prob)}, pred, prob

fitted, results, preds, probs = {}, [], {}, {}
for name, model in build_models().items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)]).fit(X_train, y_train)
    m, pred, prob = evaluate(pipe, X_test, y_test)
    fitted[name], preds[name], probs[name] = pipe, pred, prob
    results.append({"Model": name, **m})
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print(classification_report(y_test, pred, target_names=['No Hypertension', 'Hypertension']))

baseline_df = pd.DataFrame(results).set_index("Model").round(4)
baseline_df.sort_values("ROC-AUC", ascending=False)

**Interpretation:** **Gradient Boosting** is clearly the strongest baseline (accuracy ≈ 97.5%, ROC-AUC ≈ 0.998), followed by Random Forest and Logistic Regression (accuracy ≈ 92–93%, AUC ≈ 0.98). KNN and the single Decision Tree are the weakest (accuracy ≈ 82%). The gap between the linear model and Gradient Boosting indicates **non-linear interactions** between features (e.g. between `BP_History` and the lifestyle factors) that boosting can exploit.

### 8.1 Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 4.2))
cmaps = ['Blues', 'Purples', 'Oranges', 'Greens', 'Reds']
for ax, (name, pred), cm in zip(axes, preds.items(), cmaps):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=['No HTN', 'HTN']).plot(ax=ax, cmap=cm, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

### 8.2 Model comparison chart

In [ ]:
plot_df = baseline_df.reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(13, 6))
sns.barplot(x='Metric', y='Score', hue='Model', data=plot_df, palette='viridis')
plt.ylim(0, 1.05); plt.title('Baseline Model Performance Comparison'); plt.ylabel('Score')
plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

# 9. Cross-Validation & Overfitting Check

A single train/test split can be lucky or unlucky. **5-fold stratified cross-validation** on the training data gives a more stable estimate, and comparing train vs. test scores reveals overfitting.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_rows = []
for name, model in build_models().items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)])
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    cv_rows.append({"Model": name,
                    "CV Accuracy": s['test_accuracy'].mean(), "CV Accuracy Std": s['test_accuracy'].std(),
                    "CV Recall": s['test_recall'].mean(), "CV F1": s['test_f1'].mean(),
                    "CV ROC-AUC": s['test_roc_auc'].mean(), "CV ROC-AUC Std": s['test_roc_auc'].std(),
                    "Train ROC-AUC": s['train_roc_auc'].mean()})
cv_df = pd.DataFrame(cv_rows).set_index("Model").round(4)
cv_df.sort_values("CV ROC-AUC", ascending=False)

**Interpretation:** Cross-validated results confirm the ranking from the single split, with small standard deviations — the results are stable and not a lucky split.

In [ ]:
# Overfitting gap: Train ROC-AUC (CV folds) vs. held-out Test ROC-AUC
gap = pd.DataFrame({"Train ROC-AUC": cv_df["Train ROC-AUC"],
                    "CV ROC-AUC": cv_df["CV ROC-AUC"],
                    "Test ROC-AUC": baseline_df["ROC-AUC"]})
gap["Gap (Train - Test)"] = (gap["Train ROC-AUC"] - gap["Test ROC-AUC"]).round(4)
gap

**Interpretation:** Logistic Regression has essentially no train/test gap (underfits slightly but generalises well). Random Forest and Gradient Boosting reach ~1.0 AUC on training folds, which is normal for tree ensembles; their *test* AUC is almost as high, so the gap (≈0.00–0.04) is small and there is no serious overfitting. The Decision Tree and KNN show the largest gaps.

# 10. Feature Importance & Feature Selection

Two complementary views:
1. **Logistic Regression coefficients** (direction and size of the effect, on standardised features).
2. **Permutation importance** of a Random Forest, measured on a validation slice carved from the *training* data (so the test set stays untouched for the final evaluation).

In [ ]:
# --- 10.1 Logistic regression coefficients
lr_pipe = fitted["Logistic Regression"]
feat_names = lr_pipe.named_steps['prep'].get_feature_names_out()
coefs = pd.Series(lr_pipe.named_steps['clf'].coef_[0], index=feat_names).sort_values()
coefs.index = [c.split('__')[1] for c in coefs.index]

plt.figure(figsize=(8, 8))
colors = ['#D1495B' if v > 0 else '#4C9F70' for v in coefs]
coefs.plot(kind='barh', color=colors)
plt.axvline(0, color='black', lw=0.8)
plt.title('Logistic Regression Coefficients\n(red = raises hypertension risk, green = lowers it)')
plt.xlabel('Coefficient (log-odds)')
plt.tight_layout(); plt.show()

**Interpretation:** Being a past `Hypertension` patient has by far the largest coefficient, followed by family history, smoking, age, stress and salt intake (all raising risk), while longer sleep lowers it.

In [ ]:
# --- 10.2 Permutation importance on a validation slice of the training set
X_tr_in, X_val, y_tr_in, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                                  random_state=RANDOM_STATE, stratify=y_train)
rf_val = Pipeline([('prep', preprocessor),
                   ('clf', RandomForestClassifier(n_estimators=300, max_depth=8,
                                                  random_state=RANDOM_STATE, n_jobs=-1))]).fit(X_tr_in, y_tr_in)
perm = permutation_importance(rf_val, X_val, y_val, scoring='roc_auc', n_repeats=20,
                              random_state=RANDOM_STATE, n_jobs=-1)
perm_df = (pd.DataFrame({'Feature': X_val.columns, 'Importance': perm.importances_mean,
                         'Std': perm.importances_std})
           .sort_values('Importance', ascending=False).reset_index(drop=True))

plt.figure(figsize=(9, 6))
plot_perm = perm_df.iloc[::-1]
plt.barh(plot_perm['Feature'], plot_perm['Importance'], xerr=plot_perm['Std'], color='#4C9F70', capsize=3)
plt.xlabel('Importance'); plt.title('Permutation Importance (drop in ROC-AUC when feature is shuffled)')
plt.tight_layout(); plt.show()
perm_df.round(4)

**Interpretation:** `BP_History` dominates (dropping ≈0.29 AUC when shuffled), followed by `Family_History`, `Smoking_Status`, `Age`, `Salt_Intake`, `Stress_Score` and `Sleep_Duration`. `Exercise_Level`, `Poor_Sleep` and `Medication` contribute essentially nothing.

### 10.3 Feature selection

Features whose permutation importance is clearly above zero (importance > 1 standard deviation) are kept; the rest add noise. The reduced feature set is then compared against the full one using cross-validation.

In [ ]:
selected = perm_df.loc[perm_df['Importance'] > perm_df['Std'], 'Feature'].tolist()
print("Selected features:", selected)
print("Dropped features :", [f for f in X.columns if f not in selected])

sel_num = [f for f in selected if f in num_features]
sel_cat = [f for f in selected if f in cat_features]
sel_bin = [f for f in selected if f in bin_features]

def cv_auc(cols_num, cols_cat, cols_bin, model, Xtr=X_train):
    pipe = Pipeline([('prep', make_preprocessor(cols_num, cols_cat, cols_bin)), ('clf', model)])
    cols = list(cols_num) + list(cols_cat) + list(cols_bin)
    s = cross_validate(pipe, Xtr[cols], y_train, cv=cv, scoring=['roc_auc', 'accuracy'])
    return s['test_roc_auc'].mean(), s['test_accuracy'].mean()

cmp_rows = []
for name, model_fn in [("Logistic Regression", lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
                       ("Random Forest", lambda: RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)),
                       ("Gradient Boosting", lambda: GradientBoostingClassifier(random_state=RANDOM_STATE))]:
    a_all, acc_all = cv_auc(num_features, cat_features, bin_features, model_fn())
    a_sel, acc_sel = cv_auc(sel_num, sel_cat, sel_bin, model_fn())
    cmp_rows.append({"Model": name, f"AUC (all {X.shape[1]} features)": a_all, f"AUC (selected {len(selected)})": a_sel,
                     "Acc (all)": acc_all, "Acc (selected)": acc_sel})
pd.DataFrame(cmp_rows).set_index("Model").round(4)

**Interpretation:** Only `Exercise_Level` and `Poor_Sleep` are dropped (13 of 15 features kept; `Medication` just clears the bar). Dropping them leaves performance essentially unchanged, giving a slightly simpler model with no loss of accuracy.

# 11. Hyperparameter Tuning

`GridSearchCV` (Logistic Regression) and `RandomizedSearchCV` (Random Forest, Gradient Boosting) optimise **ROC-AUC** with 5-fold stratified CV, using the selected feature set. The test set is *not* used during tuning.

In [ ]:
final_cols = sel_num + sel_cat + sel_bin
X_train_s, X_test_s = X_train[final_cols], X_test[final_cols]
prep_s = make_preprocessor(sel_num, sel_cat, sel_bin)

searches = {
    "Logistic Regression": GridSearchCV(
        Pipeline([('prep', prep_s), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
        {'clf__C': [0.01, 0.1, 1, 10, 100], 'clf__penalty': ['l2'], 'clf__class_weight': [None, 'balanced']},
        scoring='roc_auc', cv=cv, n_jobs=-1),
    "Random Forest": RandomizedSearchCV(
        Pipeline([('prep', prep_s), ('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
        {'clf__n_estimators': [100, 200, 300, 500], 'clf__max_depth': [3, 4, 5, 6, 8, 10, None],
         'clf__min_samples_split': [2, 5, 10, 20], 'clf__min_samples_leaf': [1, 2, 4, 8],
         'clf__max_features': ['sqrt', 'log2', 0.5]},
        n_iter=30, scoring='roc_auc', cv=cv, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": RandomizedSearchCV(
        Pipeline([('prep', prep_s), ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        {'clf__n_estimators': [50, 100, 200, 300], 'clf__learning_rate': [0.01, 0.03, 0.05, 0.1],
         'clf__max_depth': [2, 3, 4], 'clf__subsample': [0.7, 0.85, 1.0],
         'clf__min_samples_leaf': [1, 5, 10, 20]},
        n_iter=30, scoring='roc_auc', cv=cv, random_state=RANDOM_STATE, n_jobs=-1),
}

tuned, tuned_rows = {}, []
for name, search in searches.items():
    search.fit(X_train_s, y_train)
    tuned[name] = search.best_estimator_
    m, _, _ = evaluate(search.best_estimator_, X_test_s, y_test)
    tuned_rows.append({"Model": name, "Best CV ROC-AUC": search.best_score_, **m})
    print(f"{name}: best CV AUC = {search.best_score_:.4f}")
    print("   best params:", {k.replace('clf__', ''): v for k, v in search.best_params_.items()})

tuned_df = pd.DataFrame(tuned_rows).set_index("Model").round(4)
tuned_df.sort_values("ROC-AUC", ascending=False)

**Interpretation:** Tuning mainly helps the tree ensembles. Gradient Boosting reaches ~99.5% test accuracy and Random Forest improves to ~0.99 AUC, while Logistic Regression is unchanged (a linear model is limited by its structure rather than its hyper-parameters).

In [ ]:
# Before vs. after tuning (test ROC-AUC)
before_after = pd.DataFrame({"Baseline (all features)": baseline_df.loc[tuned_df.index, "ROC-AUC"],
                             "Tuned (selected features)": tuned_df["ROC-AUC"]})
before_after.plot(kind='bar', figsize=(8, 5), rot=0)
plt.ylim(before_after.values.min() - 0.03, 1.0); plt.ylabel('Test ROC-AUC')
plt.title('Effect of Feature Selection + Hyperparameter Tuning')
plt.tight_layout(); plt.show()
before_after.round(4)

# 12. ROC Curves, Precision-Recall Curves & Threshold Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for name, mdl in tuned.items():
    prob = mdl.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, prob):.3f})")
    p, r, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(r, p, lw=2, label=f"{name} (AP = {average_precision_score(y_test, prob):.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves (tuned models)')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1, label='No-skill')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves (tuned models)')
for ax in axes: ax.legend(loc='lower right' if ax is axes[0] else 'lower left')
plt.tight_layout(); plt.show()

### 12.1 Selecting the best model
The model with the highest **cross-validated** ROC-AUC (not test AUC, to avoid selecting on the test set) is chosen as the final model.

In [ ]:
best_name = tuned_df['Best CV ROC-AUC'].idxmax()
best_model = tuned[best_name]
print(f"Final model: {best_name}")
tuned_df.loc[[best_name]]

### 12.2 Threshold analysis
The default 0.5 cut-off is not always the right one. For screening, a lower threshold increases recall (fewer missed cases) at the cost of precision. The table shows the trade-off on the **test** set for the final model.

In [ ]:
prob_best = best_model.predict_proba(X_test_s)[:, 1]
thr_rows = []
for t in np.arange(0.2, 0.81, 0.1):
    pred_t = (prob_best >= t).astype(int)
    thr_rows.append({"Threshold": round(t, 1), "Accuracy": accuracy_score(y_test, pred_t),
                     "Precision": precision_score(y_test, pred_t), "Recall": recall_score(y_test, pred_t),
                     "F1": f1_score(y_test, pred_t)})
thr_df = pd.DataFrame(thr_rows).set_index("Threshold").round(4)

thr_df[['Precision', 'Recall', 'F1']].plot(figsize=(8, 4.5), marker='o')
plt.title(f'Threshold trade-off — {best_name}'); plt.ylabel('Score'); plt.grid(True)
plt.tight_layout(); plt.show()
thr_df

**Interpretation:** At the default 0.5 threshold the model has perfect precision and 99% recall on this test set. Lowering the threshold to 0.2–0.3 pushes recall to ~99–100% at a small cost in precision — a sensible choice if the model is used for **screening**, where missing a true case is costlier than a false alarm.

# 13. Leakage / Robustness Check — Model Without `BP_History`

`BP_History` (past blood-pressure status) is by far the strongest predictor: ~93% of people with a `Hypertension` history have hypertension. A person's previous BP status is closely tied to the outcome, so a model that relies on it mostly "knows the answer already". To understand how much the **lifestyle features alone** can predict, the model is re-trained **without** `BP_History` (and `Medication`, which can be a *consequence* of diagnosis rather than a cause — although here it shows no statistical association with the target).

In [ ]:
leaky = ['BP_History', 'Medication']
abl_num = [c for c in final_cols if c in num_features]
abl_cat = [c for c in final_cols if c in cat_features and c not in leaky]
abl_bin = [c for c in final_cols if c in bin_features]
# make sure the strong lifestyle predictors are available even if feature selection dropped them
abl_num = list(dict.fromkeys(abl_num + ['Age', 'BMI', 'Salt_Intake', 'Stress_Score', 'Sleep_Duration']))
abl_cat = list(dict.fromkeys(abl_cat + ['Family_History', 'Exercise_Level', 'Smoking_Status']))
abl_cols = abl_num + abl_cat + abl_bin

abl_rows = []
for label, cols_n, cols_c, cols_b in [("With BP_History + Medication", sel_num, sel_cat, sel_bin),
                                      ("Without BP_History + Medication", abl_num, abl_cat, abl_bin)]:
    for name, mdl in [("Logistic Regression", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
                      ("Random Forest", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)),
                      ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))]:
        cols = cols_n + cols_c + cols_b
        pipe = Pipeline([('prep', make_preprocessor(cols_n, cols_c, cols_b)), ('clf', mdl)]).fit(X_train[cols], y_train)
        m, _, _ = evaluate(pipe, X_test[cols], y_test)
        abl_rows.append({"Feature set": label, "Model": name, **m})

abl_df = pd.DataFrame(abl_rows).round(4)
abl_df

**Interpretation:** Without `BP_History` the best ROC-AUC falls from ≈0.99 to ≈0.87 (accuracy ≈ 79–80%). The lifestyle factors alone (age, family history, smoking, stress, salt, sleep, BMI) therefore remain **genuinely predictive**, but the headline 97–99% accuracy depends heavily on knowing the person's blood-pressure history. If the model is meant to predict *new* hypertension risk in people without any BP record, expect performance closer to the lower figure.

In [ ]:
abl_df.pivot(index='Model', columns='Feature set', values='ROC-AUC').plot(kind='bar', figsize=(8, 5), rot=0)
plt.ylabel('Test ROC-AUC'); plt.ylim(0.5, 1.0); plt.title('Impact of removing BP_History & Medication')
plt.tight_layout(); plt.show()

# 14. Final Model Evaluation

In [ ]:
pred_best = (prob_best >= 0.5).astype(int)
print(f"FINAL MODEL: {best_name}\n")
print(classification_report(y_test, pred_best, target_names=['No Hypertension', 'Hypertension']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_best), display_labels=['No HTN', 'HTN']).plot(ax=axes[0], cmap='Greens', colorbar=False)
axes[0].set_title(f'Confusion Matrix — {best_name}')
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_best, normalize='true'), display_labels=['No HTN', 'HTN']).plot(ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f')
axes[1].set_title('Normalised (per actual class)')
plt.tight_layout(); plt.show()

## 14.1 Saving the model & predicting for new patients

In [ ]:
joblib.dump({"model": best_model, "features": final_cols}, "hypertension_model.joblib")
print("Saved -> hypertension_model.joblib")

# Load it back and predict for new, raw patient records (feature engineering applied exactly as in training)
bundle = joblib.load("hypertension_model.joblib")

def engineer(raw: pd.DataFrame) -> pd.DataFrame:
    d = raw.copy()
    d['Medication'] = d['Medication'].fillna('None')
    d['BMI_Category'] = pd.cut(d['BMI'], bins=[0, 18.5, 25, 30, 100],
                               labels=['Underweight', 'Normal', 'Overweight', 'Obese']).astype(str)
    d['Age_Group'] = pd.cut(d['Age'], bins=[17, 34, 49, 64, 120], labels=['18-34', '35-49', '50-64', '65+']).astype(str)
    d['Poor_Sleep'] = (d['Sleep_Duration'] < 6).astype(int)
    d['High_Stress'] = (d['Stress_Score'] >= 7).astype(int)
    d['Lifestyle_Risk_Score'] = ((d['Smoking_Status'] == 'Smoker').astype(int) + (d['Exercise_Level'] == 'Low').astype(int)
                                 + d['Poor_Sleep'] + d['High_Stress'] + d['BMI_Category'].isin(['Overweight', 'Obese']).astype(int))
    return d

new_patients = pd.DataFrame([
    {"Age": 68, "Salt_Intake": 11.0, "Stress_Score": 8, "BP_History": "Hypertension", "Sleep_Duration": 5.0, "BMI": 31.5,
     "Medication": "Diuretic", "Family_History": "Yes", "Exercise_Level": "Low", "Smoking_Status": "Smoker"},
    {"Age": 27, "Salt_Intake": 6.0, "Stress_Score": 2, "BP_History": "Normal", "Sleep_Duration": 8.0, "BMI": 22.0,
     "Medication": None, "Family_History": "No", "Exercise_Level": "High", "Smoking_Status": "Non-Smoker"},
])
proba = bundle["model"].predict_proba(engineer(new_patients)[bundle["features"]])[:, 1]
out = new_patients[['Age', 'BMI', 'BP_History', 'Exercise_Level']].copy()
out['P(Hypertension)'] = proba.round(3)
out['Prediction'] = np.where(proba >= 0.5, 'Hypertension', 'No Hypertension')
out

# 15. Conclusions

**Key findings**

1. **Data quality:** the only missing values were in `Medication` (blank = *no medication*), handled as an explicit `'None'` category. There were no duplicates and no problematic outliers. The target is balanced (~52/48).
2. **Strongest risk factors:** past blood-pressure status (`BP_History`) by a wide margin, followed by family history, smoking, age, salt intake, stress and (shorter) sleep. `Exercise_Level` and `Medication` showed no meaningful relationship with the outcome in this data.
3. **Model comparison:** Gradient Boosting > Random Forest ≈ Logistic Regression > KNN ≈ Decision Tree. Tuned Gradient Boosting was selected on cross-validated ROC-AUC and reaches ~99.5% accuracy on the held-out test set.
4. **Feature selection** removed 2 uninformative features with no loss of performance.
5. **Robustness check:** without `BP_History`, ROC-AUC falls to ≈0.87 — the lifestyle features alone are useful but far less decisive.

**Caveats**

* Test accuracy near 99% together with extremely high CV scores (AUC ≈ 0.9999) and the near-uniform, uncorrelated feature distributions suggest the data may be **synthetic or generated by a fairly deterministic rule**. Real-world clinical data is rarely this separable — validate on external data before drawing clinical conclusions.
* With only ~2,000 records, the test set has ~400 patients, so each misclassification moves accuracy by ~0.25 percentage points.
* This model is an educational / decision-support prototype, **not a diagnostic tool**.

**Possible next steps:** external validation, probability calibration, SHAP explanations, and trying XGBoost / LightGBM.